Question 1

In [3]:
reactions = [
    "CC(=O)O.OCC>[H+].[Cl-]>CC(=O)OCC.O",
    "C=C.[H][H]>[Pd]>CC",
    "c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O"
]

def parse_reaction(rxn):
    # Split on ">" and check there are three fields
    fields = rxn.split(">")
    if len(fields) != 3:
        raise ValueError("Reaction must contain exactly two '>' separators")

    # Split each field on "." and drop empty entries
    return [field.split(".") for field in fields]
for s in reactions :
  print(len(parse_reaction(s)))
  print(parse_reaction(s))
print("first one is esterification as it starts with acid and alchol and ends up with ester")
print("The empty space of third reaction refers to no catalyst")

3
[['CC(=O)O', 'OCC'], ['[H+]', '[Cl-]'], ['CC(=O)OCC', 'O']]
3
[['C=C', '[H][H]'], ['[Pd]'], ['CC']]
3
[['c1ccccc1', 'O=[N+]([O-])O'], [''], ['c1ccccc1[N+](=O)[O-]', 'O']]
first one is esterification as it starts with acid and alchol and ends up with ester
The empty space of third reaction refers to an intermediate


Question 2

In [5]:
import numpy as np

# columns: C2H6, O2, CO2, H2O
# rows: C, H, O
E = np.array([[2., 0., -1., 0.],
              [6., 0.,  0., -2.],
              [0., 2., -2., -1.]])

# SVD
U, s, Vt = np.linalg.svd(E)

# Rank and nullity
tol = 1e-12
rank = np.sum(s > tol)
nullity = E.shape[1] - rank

# Null vector = last row of Vt (corresponding to zero singular value)
x = Vt[-1, :]

# Rescale so C2H6 coefficient = 1
x_scaled = x / x[0]


x_whole = np.rint(x_scaled * 2).astype(int)

# Verify
check = E @ x_whole

print("Singular values:", s)
print("Rank:", rank)
print("Nullity:", nullity)
print("Coefficients scaled to C2H6 = 1:", x_scaled)
print("Smallest whole-number coefficients:", x_whole)
print("E @ x:", check)
print(" A one-dimensional null space means there is, up to an overall scaling factor, exactly one independent way to balance the reaction while conserving C, H, and O atoms.")


Singular values: [6.62561256 3.00720721 1.02857332]
Rank: 3
Nullity: 1
Coefficients scaled to C2H6 = 1: [1.  3.5 2.  3. ]
Smallest whole-number coefficients: [2 7 4 6]
E @ x: [0. 0. 0.]
 A one-dimensional null space means there is, up to an overall scaling factor, exactly one independent way to balance the reaction while conserving C, H, and O atoms.


Question 3

In [9]:
import numpy as np

# Adjacency matrix for butadiene (4 carbon atoms, open chain)
A = np.zeros((4, 4))

# Bond atom i to atom i+1
for i in range(3):
    A[i, i + 1] = 1
    A[i + 1, i] = 1

# Eigenvalues of the adjacency matrix, sorted in descending order
eigenvalues = np.linalg.eigvalsh(A)[::-1]

# Degree of each atom
degrees = np.sum(A, axis=1)

# Two lowest-energy pi MOs are occupied
# beta < 0, so we use the two largest eigenvalues
E_pi = 2 * (eigenvalues[0] + eigenvalues[1])

# Localized reference: two C=C bonds
E_localized = 4.0

# Delocalisation energy
delocalisation_energy = E_pi - E_localized

# Print results
print("Eigenvalues:", eigenvalues)
print("Degrees:", degrees)
print(f"E_pi = {E_pi:.6f} beta")
print(f"Delocalisation energy = {delocalisation_energy:.6f} beta")

# Comparison with benzene
print(
    f"Butadiene has a delocalisation energy of "
    f"{delocalisation_energy:.6f} beta, compared with benzene's 2 beta. "
    f"This shows that benzene has greater pi-electron delocalisation "
    f"and greater aromatic stabilisation."
)

Eigenvalues: [ 1.61803399  0.61803399 -0.61803399 -1.61803399]
Degrees: [1. 2. 2. 1.]
E_pi = 4.472136 beta
Delocalisation energy = 0.472136 beta
Butadiene has a delocalisation energy of 0.472136 beta, compared with benzene's 2 beta. This shows that benzene has greater pi-electron delocalisation and greater aromatic stabilisation.


Question 4

In [10]:
import numpy as np


A6 = np.zeros((6, 6))

for i in range(6):
    j = (i + 1) % 6
    A6[i, j] = 1.0
    A6[j, i] = 1.0


A_tilde = A6 + np.eye(6)


D = A_tilde.sum(axis=1)
P = A_tilde / D[:, None]


H = np.random.default_rng(1).normal(size=(6, 3))


eigenvalues = np.linalg.eigvals(P)
eigenvalue_moduli = np.sort(np.abs(eigenvalues))[::-1]

rho = eigenvalue_moduli[1]


mu = 1.0 - rho


ks = [0, 1, 2, 4, 8, 16]

deviations = []

for k in ks:
    H_k = np.linalg.matrix_power(P, k) @ H


    mean_row = H_k.mean(axis=0)


    row_deviations = np.linalg.norm(H_k - mean_row, axis=1)


    max_deviation = np.max(row_deviations)

    deviations.append(max_deviation)

print("Moduli of all six eigenvalues of P:")
print(eigenvalue_moduli)

print("\nSmoothing rate mu:")
print(mu)

print("\nLargest deviation from mean row:")
for k, deviation in zip(ks, deviations):
    print(f"k = {k:2d} : {deviation:.6f}")

print(
    "\nImplication: For a molecule with about a dozen heavy atoms, "
    "roughly 4–8 graph-convolution layers should provide substantial "
    "information mixing while avoiding excessive oversmoothing; "
    "16 layers is already very close to the mean representation."
)

Moduli of all six eigenvalues of P:
[1.00000000e+00 6.66666667e-01 6.66666667e-01 3.33333333e-01
 7.91848035e-17 3.92523115e-17]

Smoothing rate mu:
0.33333333333333326

Largest deviation from mean row:
k =  0 : 1.241388
k =  1 : 0.536825
k =  2 : 0.345910
k =  4 : 0.154873
k =  8 : 0.030668
k = 16 : 0.001197

Implication: For a molecule with about a dozen heavy atoms, roughly 4–8 graph-convolution layers should provide substantial information mixing while avoiding excessive oversmoothing; 16 layers is already very close to the mean representation.


Question 5

In [14]:
import numpy as np

X = np.array([
    [78.1, 2.3],
    [92.1, 1.7],
    [106.2, 2.8],
    [120.2, 2.0],
    [134.2, 3.1],
    [148.2, 2.4]
])
def pca(M):
    C = M.T@M/len(M)
    w,V = np.linalg.eigh(C)
    i = np.argsort(w)[::-1]
    w,V = w[i], V[:,i]
    V *= np.sign(V[abs(V).argmax(0), [0,1]])
    print(C, w, w/w.sum(), V[:,0])


Xc = X - X.mean(0)
pca(Xc)
print()
pca(Xc/Xc.std(0))
print()

print("Molar mass has a vey high variance , so unstandardised PCA just recovering the mass axis whereas the standardised one  result  to report.")



[[5.73535556e+02 4.56277778e+00]
 [4.56277778e+00 2.18055556e-01]] [5.73571866e+02 1.81744746e-01] [9.99683236e-01 3.16764448e-04] [0.99996834 0.0079578 ]

[[1.         0.40800508]
 [0.40800508 1.        ]] [1.40800508 0.59199492] [0.70400254 0.29599746] [0.70710678 0.70710678]

Molar mass has a vey high variance , so unstandardised PCA just recovering the mass axis whereas the standardised one  result  to report.
